# RETFound DR — Next Steps (Analysis + Ablations + External Test)

Run **after** M0/M2 training. Assumes:
- Same session **or** checkpoints in `CFG.OUT_DIR` / a Kaggle dataset you saved
- APTOS splits already defined (`train_df`, `val_df`, `test_df`) **or** reload from `splits.json`

**Order:** 1) Reload preds → 2) CM + per-class → 3) Bootstrap/McNemar → 4) Ablations → 5) External test → 6) Seeds (optional)

In [ ]:
# ========================= NEXT-STEP CONFIG =========================
# Reuse CFG from main notebook if present; else minimal defaults
from pathlib import Path
import os, json, math, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import (
    confusion_matrix, classification_report, cohen_kappa_score,
    accuracy_score, f1_score, roc_auc_score
)
from scipy.stats import chi2
import matplotlib.pyplot as plt

OUT = Path(CFG.OUT_DIR) if "CFG" in dir() else Path("/kaggle/working/outputs")
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = CFG.DEVICE if "CFG" in dir() else ("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = CFG.NUM_CLASSES if "CFG" in dir() else 5

# Set True only for sections you want to run
RUN_RELOAD_PREDS = True
RUN_CM_ANALYSIS = True
RUN_STATS = True
RUN_ABLATIONS = False   # expensive — turn on when ready
RUN_EXTERNAL = False    # needs external dataset attached
RUN_SEEDS = False       # expensive

ABLATION_EPOCHS = 15    # shorter than full M2 for ablations
EXTERNAL_DIR = "/kaggle/input/messidor2"  # change to your dataset path
EXTERNAL_CSV = None     # auto-detect if None
SEEDS = [42, 43, 44]

print("OUT:", OUT)
print("DEVICE:", DEVICE)

## 1) Reload best checkpoints & test predictions
Skip if `yt0, yp0, yt2, yp2` already exist in memory from training.

In [ ]:
@torch.no_grad()
def predict_model(model, loader, model_name="M0"):
    model.eval()
    ys, preds, probs = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        if model_name == "M0":
            logits = out
        else:
            logits = out["logits"]
        pr = torch.softmax(logits, dim=-1)
        pred = logits.argmax(dim=-1)
        ys.append(y.numpy())
        preds.append(pred.cpu().numpy())
        probs.append(pr.cpu().numpy())
    return np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)


def load_m0_best(path=OUT / "M0_best.pt"):
    model = M0RetFound(NUM_CLASSES, WEIGHTS_PATH).to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False)
    model.eval()
    return model


def load_m2_best(path=OUT / "M2_best.pt"):
    model = M2RetFound(
        num_classes=NUM_CLASSES,
        weights_path=WEIGHTS_PATH,
        ms_blocks=CFG.MS_BLOCKS,
        lora_r=CFG.LORA_R,
        lora_alpha=CFG.LORA_ALPHA,
        lora_dropout=CFG.LORA_DROPOUT,
        lora_targets=CFG.LORA_TARGETS,
    ).to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False)
    model.eval()
    return model


if RUN_RELOAD_PREDS:
    need = not ("yt0" in dir() and "yp0" in dir() and "yt2" in dir() and "yp2" in dir())
    if need:
        print("Reloading checkpoints and predicting on test_loader...")
        assert "test_loader" in dir(), "test_loader missing — re-run data cells from main notebook"
        m0_best = load_m0_best()
        m2_best = load_m2_best()
        yt0, yp0, ypr0 = predict_model(m0_best, test_loader, "M0")
        yt2, yp2, ypr2 = predict_model(m2_best, test_loader, "M2")
    else:
        print("Using in-memory predictions yt*/yp*")
        # ensure probs exist for AUROC bootstrap
        if "ypr0" not in dir() or "ypr2" not in dir():
            m0_best = load_m0_best()
            m2_best = load_m2_best()
            _, _, ypr0 = predict_model(m0_best, test_loader, "M0")
            _, _, ypr2 = predict_model(m2_best, test_loader, "M2")
    assert np.array_equal(yt0, yt2), "M0/M2 test labels differ — check loaders/splits"
    y_true = yt0
    print("Test N=", len(y_true))

## 2) Confusion matrices + per-class report

In [ ]:
if RUN_CM_ANALYSIS:
    def plot_cm(y_true, y_pred, title, save_path):
        cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
        fig, ax = plt.subplots(figsize=(5, 4.5))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_title(title)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        fig.savefig(save_path, dpi=150)
        plt.show()
        return cm

    cm0 = plot_cm(y_true, yp0, "M0 Test CM", OUT / "M0_cm_next.png")
    cm2 = plot_cm(y_true, yp2, "M2 Test CM", OUT / "M2_cm_next.png")

    print("===== M0 per-class =====")
    print(classification_report(y_true, yp0, digits=4))
    print("===== M2 per-class =====")
    print(classification_report(y_true, yp2, digits=4))

    # Adjacent vs distant errors (ordinal)
    def error_profile(y_t, y_p):
        d = np.abs(y_t - y_p)
        return {
            "exact": float((d == 0).mean()),
            "off_by_1": float((d == 1).mean()),
            "off_by_ge2": float((d >= 2).mean()),
            "mean_abs_error": float(d.mean()),
        }

    prof = pd.DataFrame([
        {"model": "M0", **error_profile(y_true, yp0)},
        {"model": "M2", **error_profile(y_true, yp2)},
    ])
    display(prof)
    prof.to_csv(OUT / "error_profile.csv", index=False)

## 3) Statistical tests — Bootstrap CI on ΔQWK + McNemar

In [ ]:
if RUN_STATS:
    def qwk(y_t, y_p):
        return cohen_kappa_score(y_t, y_p, weights="quadratic")

    def bootstrap_delta_qwk(y_t, y_a, y_b, n_boot=2000, seed=42):
        rng = np.random.default_rng(seed)
        n = len(y_t)
        deltas = []
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            deltas.append(qwk(y_t[idx], y_b[idx]) - qwk(y_t[idx], y_a[idx]))
        deltas = np.asarray(deltas)
        lo, hi = np.percentile(deltas, [2.5, 97.5])
        return {
            "delta_qwk": float(qwk(y_t, y_b) - qwk(y_t, y_a)),
            "ci95_low": float(lo),
            "ci95_high": float(hi),
            "ci_excludes_0": bool(lo > 0 or hi < 0),
        }

    def mcnemar_referable(y_t, y_a, y_b):
        # Compare binary referable decisions (grade >= 2)
        t = (y_t >= 2).astype(int)
        a = (y_a >= 2).astype(int)
        b = (y_b >= 2).astype(int)
        a_correct = a == t
        b_correct = b == t
        b01 = int(np.sum(~a_correct & b_correct))  # M0 wrong, M2 right
        b10 = int(np.sum(a_correct & ~b_correct))  # M0 right, M2 wrong
        # Continuity-corrected McNemar
        stat = (abs(b01 - b10) - 1) ** 2 / max(b01 + b10, 1)
        p = float(chi2.sf(stat, 1))
        return {"b01_M2_fixes": b01, "b10_M2_breaks": b10, "stat": float(stat), "p_value": p}

    stats_qwk = bootstrap_delta_qwk(y_true, yp0, yp2, n_boot=2000)
    stats_mc = mcnemar_referable(y_true, yp0, yp2)

    # Bootstrap referable AUROC delta
    def ref_auroc(y_t, prob):
        yt = (y_t >= 2).astype(int)
        score = prob[:, 2:].sum(axis=1)
        return roc_auc_score(yt, score)

    rng = np.random.default_rng(0)
    n = len(y_true)
    d_auc = []
    for _ in range(2000):
        idx = rng.integers(0, n, n)
        try:
            d_auc.append(ref_auroc(y_true[idx], ypr2[idx]) - ref_auroc(y_true[idx], ypr0[idx]))
        except ValueError:
            pass
    d_auc = np.asarray(d_auc)
    stats_auc = {
        "delta_auroc": float(ref_auroc(y_true, ypr2) - ref_auroc(y_true, ypr0)),
        "ci95_low": float(np.percentile(d_auc, 2.5)),
        "ci95_high": float(np.percentile(d_auc, 97.5)),
    }

    print("QWK M0:", qwk(y_true, yp0), "| M2:", qwk(y_true, yp2))
    print("ΔQWK (M2-M0) + 95% CI:", stats_qwk)
    print("Referable McNemar:", stats_mc)
    print("ΔAUROC (M2-M0) + 95% CI:", stats_auc)

    Path(OUT / "stats.json").write_text(json.dumps({
        "qwk": stats_qwk, "mcnemar_referable": stats_mc, "auroc": stats_auc,
        "m0_qwk": float(qwk(y_true, yp0)), "m2_qwk": float(qwk(y_true, yp2)),
    }, indent=2))
    print("Saved", OUT / "stats.json")

## 4) Ablations (same split, shorter epochs)

| ID | Variant |
|----|---------|
| A1 | LoRA + CLS head + focal only |
| A2 | LoRA + multi-scale + focal only |
| A3 | Full M2 (multi-scale + focal + ordinal + referable) |

Set `RUN_ABLATIONS = True` above. Reuses `train_loader` / `val_loader` / `test_loader`.

In [ ]:
class M2Ablation(nn.Module):
    """Configurable M2 for ablations."""
    def __init__(self, use_multiscale=True, num_classes=5, weights_path="",
                 ms_blocks=(7, 15, 23), lora_r=8, lora_alpha=16, lora_dropout=0.05):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.ms_blocks = tuple(ms_blocks) if use_multiscale else (23,)
        self.backbone = load_vit_backbone(num_classes=0, weights_path=weights_path)
        for p in self.backbone.parameters():
            p.requires_grad = False
        inject_lora_timm_vit(self.backbone, r=lora_r, alpha=lora_alpha, dropout=lora_dropout, target_names=("qkv",))
        dim = getattr(self.backbone, "embed_dim", 1024)
        n_scales = len(self.ms_blocks)
        self.head = MultiScaleFusionHead(dim=dim, num_classes=num_classes, n_scales=n_scales)
        self._hooks, self._cache = [], {}
        self._register_hooks()

    def _register_hooks(self):
        blocks = self.backbone.blocks
        def make_hook(i):
            def hook(_m, _i, out):
                self._cache[i] = 0.5 * (out[:, 0] + out[:, 1:].mean(dim=1))
            return hook
        for i in self.ms_blocks:
            self._hooks.append(blocks[i].register_forward_hook(make_hook(i)))

    def forward(self, x):
        self._cache = {}
        _ = self.backbone.forward_features(x)
        feats = [self._cache[i] for i in self.ms_blocks]
        return self.head(feats)


def train_ablation(name, use_multiscale=True, use_ordinal=True, use_referable=True, epochs=None):
    epochs = epochs or ABLATION_EPOCHS
    model = M2Ablation(
        use_multiscale=use_multiscale,
        num_classes=NUM_CLASSES,
        weights_path=WEIGHTS_PATH,
        ms_blocks=CFG.MS_BLOCKS,
        lora_r=CFG.LORA_R,
        lora_alpha=CFG.LORA_ALPHA,
        lora_dropout=CFG.LORA_DROPOUT,
    ).to(DEVICE)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=CFG.LR_M2, weight_decay=CFG.WEIGHT_DECAY)
    focal = FocalLoss(gamma=CFG.FOCAL_GAMMA, weight=CW)
    coral = CoralOrdinalLoss(NUM_CLASSES)
    bce = nn.BCEWithLogitsLoss()

    best = {"qwk": -1, "path": str(OUT / f"{name}_best.pt")}
    for epoch in range(epochs):
        model.train()
        cosine_lr(opt, epoch, epochs, CFG.LR_M2, min(2, epochs // 5 or 1))
        losses = []
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss = focal(out["logits"], y)
            if use_ordinal:
                loss = loss + 0.5 * coral(out["ordinal"], y)
            if use_referable:
                loss = loss + 0.3 * bce(out["referable"], (y >= 2).float())
            loss.backward(); opt.step()
            losses.append(loss.item())
        val_m, *_ = evaluate(model, val_loader, "M2")
        print(f"[{name}][{epoch}] loss={np.mean(losses):.4f} val_qwk={val_m['qwk']:.4f}")
        if val_m["qwk"] > best["qwk"]:
            best["qwk"] = val_m["qwk"]
            torch.save({"model": model.state_dict(), "val": val_m}, best["path"])

    ckpt = torch.load(best["path"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False)
    test_m, yt, yp, ypr = evaluate(model, test_loader, "M2")
    print(f"[{name}] TEST", {k: test_m[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")})
    return test_m


if RUN_ABLATIONS:
    abl_rows = []
    # Include already-trained full models if available
    if "results" in dir() and "M0" in results:
        t = results["M0"]["test"]
        abl_rows.append({"variant": "M0_fullFT", **{k: t[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}})
    if "results" in dir() and "M2" in results:
        t = results["M2"]["test"]
        abl_rows.append({"variant": "M2_full", **{k: t[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}})

    configs = [
        ("A1_lora_cls_focal", dict(use_multiscale=False, use_ordinal=False, use_referable=False)),
        ("A2_lora_ms_focal", dict(use_multiscale=True, use_ordinal=False, use_referable=False)),
        ("A3_lora_ms_ord_ref", dict(use_multiscale=True, use_ordinal=True, use_referable=True)),
    ]
    for name, kw in configs:
        print("\n=====", name, "=====")
        tm = train_ablation(name, **kw)
        abl_rows.append({"variant": name, **{k: tm[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}})

    abl_df = pd.DataFrame(abl_rows)
    display(abl_df)
    abl_df.to_csv(OUT / "ablation_results.csv", index=False)
else:
    print("Ablations skipped (RUN_ABLATIONS=False)")

## 5) External test (no retraining)

1. Add Messidor-2 / DDR as a Kaggle dataset  
2. Set `EXTERNAL_DIR` / CSV columns below  
3. Set `RUN_EXTERNAL = True`

Expected CSV columns (rename in cell if needed): `image_id` or `image`, `label` or `diagnosis` (0–4).

In [ ]:
def find_ext_image(image_id, roots):
    for root in roots:
        for ext in (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".PNG", ".JPG"):
            p = Path(root) / f"{image_id}{ext}"
            if p.is_file():
                return str(p)
            # sometimes files already include extension in id
            p2 = Path(root) / str(image_id)
            if p2.is_file():
                return str(p2)
    return None


def load_external_df(external_dir, csv_path=None):
    external_dir = Path(external_dir)
    assert external_dir.exists(), f"Missing {external_dir}"
    if csv_path is None:
        cands = list(external_dir.rglob("*.csv"))
        assert cands, "No CSV found — set EXTERNAL_CSV"
        csv_path = cands[0]
        print("Using CSV:", csv_path)
    df = pd.read_csv(csv_path)
    # normalize columns
    colmap = {}
    for c in df.columns:
        cl = c.lower()
        if cl in ("id_code", "image_id", "image", "file", "filename", "id"):
            colmap[c] = "image_id"
        if cl in ("diagnosis", "label", "level", "dr", "grade", "adjudicated_dr_grade"):
            colmap[c] = "label"
    df = df.rename(columns=colmap)
    assert "image_id" in df.columns and "label" in df.columns, df.columns.tolist()
    df["image_id"] = df["image_id"].astype(str).str.replace(r"\.(png|jpg|jpeg|tif|tiff)$", "", regex=True)
    df["label"] = df["label"].astype(int)

    img_roots = [p for p in external_dir.rglob("*") if p.is_dir()]
    img_roots = [external_dir] + img_roots
    df["path"] = df["image_id"].apply(lambda i: find_ext_image(i, img_roots))
    print("External rows:", len(df), "missing files:", df["path"].isna().sum())
    df = df.dropna(subset=["path"]).reset_index(drop=True)
    return df


if RUN_EXTERNAL:
    ext_df = load_external_df(EXTERNAL_DIR, EXTERNAL_CSV)
    ext_ds = FundusDataset(ext_df, train=False, img_size=CFG.IMG_SIZE)
    ext_loader = DataLoader(ext_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

    m0_best = load_m0_best()
    m2_best = load_m2_best()
    yt_e0, yp_e0, ypr_e0 = predict_model(m0_best, ext_loader, "M0")
    yt_e2, yp_e2, ypr_e2 = predict_model(m2_best, ext_loader, "M2")

    def pack(yt, yp, ypr):
        m = compute_metrics(yt, yp, ypr)
        return {k: m[k] for k in ("accuracy", "macro_f1", "qwk", "referable_acc", "referable_auroc")}

    ext_cmp = pd.DataFrame([
        {"model": "M0", **pack(yt_e0, yp_e0, ypr_e0)},
        {"model": "M2", **pack(yt_e2, yp_e2, ypr_e2)},
    ])
    print("===== EXTERNAL TEST =====")
    display(ext_cmp)
    ext_cmp.to_csv(OUT / "comparison_external.csv", index=False)
    plot_cm(yt_e0, yp_e0, "M0 External CM", OUT / "M0_cm_external.png")
    plot_cm(yt_e2, yp_e2, "M2 External CM", OUT / "M2_cm_external.png")
else:
    print("External test skipped (RUN_EXTERNAL=False)")

## 6) Multi-seed (optional)
Re-trains M0+M2 for each seed — only after ablations/external plan is clear. Very GPU-heavy.

In [ ]:
if RUN_SEEDS:
    seed_rows = []
    for seed in SEEDS:
        print("\n########## SEED", seed, "##########")
        seed_everything(seed)
        CFG.SEED = seed
        # rebuild stratified split
        tr, tmp = train_test_split(df, test_size=(CFG.VAL_RATIO + CFG.TEST_RATIO), stratify=df["label"], random_state=seed)
        rel = CFG.TEST_RATIO / (CFG.VAL_RATIO + CFG.TEST_RATIO)
        va, te = train_test_split(tmp, test_size=rel, stratify=tmp["label"], random_state=seed)
        global train_df, val_df, test_df, train_loader, val_loader, test_loader, CW
        train_df, val_df, test_df = tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)
        train_loader, val_loader, test_loader = make_loaders()
        CW = class_weights_from_df(train_df, NUM_CLASSES, DEVICE)

        _, _, t0, _, _ = train_m0()
        _, _, t2, _, _ = train_m2()
        seed_rows.append({"seed": seed, "model": "M0", **{k: t0[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}})
        seed_rows.append({"seed": seed, "model": "M2", **{k: t2[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}})

    seed_df = pd.DataFrame(seed_rows)
    summary = seed_df.groupby("model")[["accuracy", "macro_f1", "qwk", "referable_auroc"]].agg(["mean", "std"])
    display(seed_df)
    display(summary)
    seed_df.to_csv(OUT / "seed_results.csv", index=False)
    summary.to_csv(OUT / "seed_summary.csv")
else:
    print("Multi-seed skipped (RUN_SEEDS=False)")

## Suggested order on Kaggle

1. Same session as training (or Save Version → download `outputs`, re-upload as dataset).  
2. Run sections **1–3 today** (cheap, minutes).  
3. Run **4 Ablations** overnight (`RUN_ABLATIONS=True`).  
4. Attach Messidor-2, set paths, `RUN_EXTERNAL=True`.  
5. Only if committee wants variance: `RUN_SEEDS=True`.